In [7]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
# from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

# --- Load API Key ---
load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
property_text = """
Riverside Retail Center is a 25,000 SF retail center
located in Austin, Texas.

The property is currently 95% occupied and is
anchored by Starbucks.

The asking price is $8.5 million.
"""

In [3]:
class Property(BaseModel):
    property_name: str
    city: str
    property_type: str
    square_feet: int = Field(gt=0)
    occupancy: int = Field(ge=0, le=100)
    price: str
    anchor_tenant: str

In [4]:
parser = PydanticOutputParser(
    pydantic_object=Property
)

In [5]:
prompt = PromptTemplate(
    template="""
Extract property information.

{format_instructions}

Property Description:
{text}
""",
    input_variables=["text"],
    partial_variables={
        "format_instructions":
        parser.get_format_instructions()
    }
)

In [8]:
llm = ChatOpenAI(model="gpt-4o-mini")

chain = prompt | llm | parser

In [9]:
result = chain.invoke({
    "text": property_text
})

result

Property(property_name='Riverside Retail Center', city='Austin', property_type='Retail', square_feet=25000, occupancy=95, price='$8.5 million', anchor_tenant='Starbucks')

In [10]:
result.model_dump()

{'property_name': 'Riverside Retail Center',
 'city': 'Austin',
 'property_type': 'Retail',
 'square_feet': 25000,
 'occupancy': 95,
 'price': '$8.5 million',
 'anchor_tenant': 'Starbucks'}

In [11]:
import pandas as pd

df = pd.DataFrame([
    result.model_dump()
])

df

,property_name,city,property_type,square_feet,occupancy,price,anchor_tenant
0,Riverside Retail Center,Austin,Retail,25000,95,$8.5 million,Starbucks
